In [ ]:
  !pip install pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir(("/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/GLAUCOMA-DETECTION-master")))

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import Dense
classifier = Sequential()
classifier.add(Conv2D(32, (3, 3), input_shape = (256,256, 3), activation = 'relu'))
classifier.add(MaxPooling2D(pool_size = (2, 2)))
classifier.add(Conv2D(32, (3, 3), activation = 'relu'))
classifier.add(MaxPooling2D(pool_size = (2, 2)))
classifier.add(Flatten())
classifier.add(Dense(units = 128, activation = 'relu'))
classifier.add(Dense(units = 1, activation = 'sigmoid'))
classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale = 1./255,
shear_range = 0.2,
zoom_range = 0.2,
horizontal_flip = True)
test_datagen = ImageDataGenerator(rescale = 1./255)
training_set = train_datagen.flow_from_directory('/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/GLAUCOMA-DETECTION-master/dataset/train',
target_size = (256,256),
batch_size = 32,
class_mode = 'binary')
test_set = test_datagen.flow_from_directory('/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/GLAUCOMA-DETECTION-master/dataset/test',
target_size = (256,256),
batch_size = 32,
class_mode = 'binary')

In [ ]:
batch_size=32
model_info = classifier.fit(
    training_set,
    steps_per_epoch=int(455 / batch_size),
    epochs=30,
    validation_data=test_set,
    validation_steps=int(30 / batch_size)
)

In [ ]:
score = classifier.evaluate(test_set, steps=int(40 / batch_size))
print(" Total: ", len(test_set.filenames))
print("Loss: ", score[0], "Accuracy: ", score[1])
print("Accuracy = ",score[1])

In [ ]:
classifier.save('/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/Colab_Project/glaucoma/Gmodel.h5')

In [ ]:
import os
from keras.models import load_model
from PIL import Image
from keras.preprocessing import image
import numpy as np
import cv2

In [ ]:
target_size = (256,256)
model=load_model('/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/Colab_Project/glaucoma/Gmodel.h5')
print("model loaded")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown
from PIL import Image
from keras.preprocessing import image
+
img_path = '/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/GLAUCOMA-DETECTION-master/dataset/test/class1/Im455.jpg'
test_image = image.load_img(img_path, target_size = (256,256))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = model.predict(test_image)
training_set.class_indices
img=Image.open(img_path)
plt.imshow(img)
plt.axis('off')
plt.show()
if result[0][0] == 1:
 print_bold_large("Glaucoma")
else:
 print_bold_large("Not Glaucoma")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown
from PIL import Image
from keras.preprocessing import image
def print_bold_large(text):
  indent = "&nbsp;" * 35
  display(Markdown(f"{indent}**<font size=5>{text}</font>**"))
img2_path = '/content/drive/My Drive/AIRTON_GLAUCOMA_DATASET/GLAUCOMA-DETECTION-master/dataset/test/class0/Im181.jpg'
test_image = image.load_img(img2_path, target_size = (256,256))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = model.predict(test_image)
training_set.class_indices
img=Image.open(img2_path)
plt.imshow(img)
plt.axis('off')
plt.show()
if result[0][0] == 1:
 print_bold_large("Glaucoma")
else:
 print_bold_large("Not Glaucoma")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline
plt.style.use('fivethirtyeight')

def plot_model_history(model_history):
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    if 'binary_accuracy' in model_history.history:
        accuracy_key = 'binary_accuracy'
    elif 'accuracy' in model_history.history:
        accuracy_key = 'accuracy'
    elif 'acc' in model_history.history:
        accuracy_key = 'acc'
    else:
        raise KeyError("None of 'binary_accuracy', 'accuracy', or 'acc' found in history keys.")

    axs[0].plot(range(1, len(model_history.history[accuracy_key]) + 1), model_history.history[accuracy_key])
    if 'val_acc' in model_history.history:
        axs[0].plot(range(1, len(model_history.history['val_acc']) + 1), model_history.history['val_acc'])
    elif 'val_accuracy' in model_history.history:
        axs[0].plot(range(1, len(model_history.history['val_accuracy']) + 1), model_history.history['val_accuracy'])
    else:
        print("Validation accuracy not found in history. Skipping validation accuracy plot.")
    axs[0].set_title('Model Accuracy')
    axs[0].set_ylabel('Accuracy')
    axs[0].set_xlabel('Epoch')
    num_epochs = len(model_history.history[accuracy_key])
    step_size = int(num_epochs / 10)
    if step_size > 0:
        axs[0].set_xticks(np.arange(1, num_epochs + 1, step_size))
    else:
        axs[0].set_xticks(np.arange(1, num_epochs + 1))

    axs[0].legend(['train', 'val'], loc='best')
    axs[1].plot(range(1, len(model_history.history['loss']) + 1), model_history.history['loss'])
    axs[1].plot(range(1, len(model_history.history['val_loss']) + 1), model_history.history['val_loss'])
    axs[1].set_title('Model Loss')
    axs[1].set_ylabel('Loss')
    axs[1].set_xlabel('Epoch')
    num_epochs = len(model_history.history['loss'])
    step_size = int(num_epochs / 10)

In [ ]:
plot_model_history(model_info)